# Playground Series: Predicting Electric Vehicle Purchases
## Advanced ROC AUC Submission Blending & Ensembling Pipeline

This notebook performs statistical analysis and ensembling of two candidate model submissions:
- **Submission 1 (`ev-blend-1.csv`)**: High-accuracy model predictions in calibrated probability space.
- **Submission 2 (`ev-blend-2.csv`)**: High-accuracy model predictions in rank-transformed uniform percentile space.

---

### Ensembling Theory for ROC AUC Competitions
In competitions evaluated on **ROC AUC (Area Under ROC Curve)**:
1. **ROC AUC is purely rank-order dependent**: It measures the probability that a randomly chosen positive instance is ranked higher than a randomly chosen negative instance. The actual numeric values do not matter, only their **relative ordering**.
2. **The Danger of Naive Linear Averaging ($0.5 \times S_1 + 0.5 \times S_2$)**:
   - `Submission 1` has an empirical probability distribution (mean $\approx 0.435$, skewed).
   - `Submission 2` is uniformly distributed in $[0, 1]$ (mean $= 0.500$, std $= 1/\sqrt{12} \approx 0.2887$).
   - A direct linear average creates severe distortion in the tails (near 0 and 1), causing ranking inversions and dropping leaderboard score.
3. **The Optimal Solution**:
   - **Normalized Rank Averaging**: Transform both submissions to uniform percentile ranks $r_i = \text{rank}(S_i) / N$, then blend:
     $$r_{\text{blend}} = w \cdot r_1 + (1 - w) \cdot r_2$$
   - **Rank-Logit Averaging**: Map ranks to unbounded log-odds $\text{logit}(r) = \ln\left(\frac{r}{1-r}\right)$, take a linear combination in logit space, and map back with the sigmoid function $\sigma(z) = \frac{1}{1 + e^{-z}}$. This provides smooth discrimination at extreme thresholds.


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, pearsonr, rankdata
from scipy.special import logit, expit

# Configure plot aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print("[INFO] Libraries imported successfully!")


## 1. Smart Input Path Auto-Detection
This block automatically resolves the paths to `ev-blend-1.csv` and `ev-blend-2.csv` whether running on **Kaggle** (e.g. `/kaggle/input/...`) or **locally** (e.g. `/home/arch/ipynb/lb/ev-blend-1.csv` and `/home/arch/ipynb/lb/ev-blend-2.csv`).


In [ ]:
def find_submission_files():
    sub1_candidates = [
        "/home/arch/ipynb/lb/ev-blend-1.csv",
        "ev-blend-1.csv",
        "./ev-blend-1.csv",
        "../ev-blend-1.csv",
        # Backward compatibility fallbacks
        "results/submission1.csv",
        "submission1.csv",
    ]
    sub1_globs = [
        "/kaggle/input/**/ev-blend-1*.csv",
        "../input/**/ev-blend-1*.csv",
        "/kaggle/working/**/ev-blend-1*.csv",
        "**/ev-blend-1*.csv",
        # Backward compatibility globs
        "/kaggle/input/**/submission1*.csv",
        "/kaggle/input/*/submission1*.csv",
        "../input/*/submission1*.csv",
        "/kaggle/working/submission1.csv",
    ]
    
    sub2_candidates = [
        "/home/arch/ipynb/lb/ev-blend-2.csv",
        "ev-blend-2.csv",
        "./ev-blend-2.csv",
        "../ev-blend-2.csv",
        # Backward compatibility fallbacks
        "results1/submission2.csv",
        "submission2.csv",
    ]
    sub2_globs = [
        "/kaggle/input/**/ev-blend-2*.csv",
        "../input/**/ev-blend-2*.csv",
        "/kaggle/working/**/ev-blend-2*.csv",
        "**/ev-blend-2*.csv",
        # Backward compatibility globs
        "/kaggle/input/**/submission2*.csv",
        "/kaggle/input/*/submission2*.csv",
        "../input/*/submission2*.csv",
        "/kaggle/working/submission2.csv",
    ]
    
    def resolve(candidates, globs):
        for path in candidates:
            if os.path.exists(path):
                return path
        for pat in globs:
            matches = glob.glob(pat, recursive=True)
            if matches:
                return sorted(matches)[0]
        return None

    sub1_path = resolve(sub1_candidates, sub1_globs)
    sub2_path = resolve(sub2_candidates, sub2_globs)
    return sub1_path, sub2_path

sub1_path, sub2_path = find_submission_files()
print(f"[INFO] Submission 1 Path: {sub1_path}")
print(f"[INFO] Submission 2 Path: {sub2_path}")

if not sub1_path or not os.path.exists(sub1_path):
    raise FileNotFoundError("Could not find ev-blend-1.csv! Please provide the correct file path or attach the dataset.")
if not sub2_path or not os.path.exists(sub2_path):
    raise FileNotFoundError("Could not find ev-blend-2.csv! Please provide the correct file path or attach the dataset.")


## 2. Load and Validate Submissions
Verify shape, column names, ID alignment, and data cleanliness.


In [ ]:
df1 = pd.read_csv(sub1_path)
df2 = pd.read_csv(sub2_path)

print(f"Submission 1 shape: {df1.shape}")
print(f"Submission 2 shape: {df2.shape}")

# Rigorous checks
assert df1.shape == df2.shape, f"Shape mismatch: {df1.shape} vs {df2.shape}"
assert (df1['id'] == df2['id']).all(), "ID mismatch between submissions!"
assert df1['Will_Buy_EV'].isna().sum() == 0, "Submission 1 contains NaN values!"
assert df2['Will_Buy_EV'].isna().sum() == 0, "Submission 2 contains NaN values!"

print(f"[PASS] Data validation passed! 286,571 test records matching from id={df1['id'].min()} to id={df1['id'].max()}.")


## 3. Statistical Profiling & Correlation Analysis
Let's examine the distributions, central tendencies, and correlations between both submissions.


In [ ]:
stats_summary = pd.DataFrame({
    'Metric': ['Count', 'Mean', 'Std Dev', 'Min', '25% Quantile', 'Median (50%)', '75% Quantile', '90% Quantile', '99% Quantile', 'Max'],
    'Submission 1 (Prob Space)': [
        f"{len(df1):,}",
        f"{df1['Will_Buy_EV'].mean():.6f}",
        f"{df1['Will_Buy_EV'].std():.6f}",
        f"{df1['Will_Buy_EV'].min():.6e}",
        f"{df1['Will_Buy_EV'].quantile(0.25):.6f}",
        f"{df1['Will_Buy_EV'].median():.6f}",
        f"{df1['Will_Buy_EV'].quantile(0.75):.6f}",
        f"{df1['Will_Buy_EV'].quantile(0.90):.6f}",
        f"{df1['Will_Buy_EV'].quantile(0.99):.6f}",
        f"{df1['Will_Buy_EV'].max():.6f}",
    ],
    'Submission 2 (Rank Space)': [
        f"{len(df2):,}",
        f"{df2['Will_Buy_EV'].mean():.6f}",
        f"{df2['Will_Buy_EV'].std():.6f}",
        f"{df2['Will_Buy_EV'].min():.6e}",
        f"{df2['Will_Buy_EV'].quantile(0.25):.6f}",
        f"{df2['Will_Buy_EV'].median():.6f}",
        f"{df2['Will_Buy_EV'].quantile(0.75):.6f}",
        f"{df2['Will_Buy_EV'].quantile(0.90):.6f}",
        f"{df2['Will_Buy_EV'].quantile(0.99):.6f}",
        f"{df2['Will_Buy_EV'].max():.6f}",
    ]
})

print(stats_summary.to_string(index=False))

p_corr, _ = pearsonr(df1['Will_Buy_EV'], df2['Will_Buy_EV'])
s_corr, _ = spearmanr(df1['Will_Buy_EV'], df2['Will_Buy_EV'])

print(f"\nPearson Linear Correlation:      {p_corr:.7f}")
print(f"Spearman Rank-Order Correlation: {s_corr:.7f}")


### Visual Comparison: Probability Distribution vs Uniform Percentile Ranks
Notice how Submission 1 has the natural shape of positive EV adoption probability, whereas Submission 2 is uniformly distributed across $[0, 1]$.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Histograms
axes[0].hist(df1['Will_Buy_EV'], bins=100, alpha=0.6, label='Submission 1 (Prob Space)', color='#1f77b4', density=True)
axes[0].hist(df2['Will_Buy_EV'], bins=100, alpha=0.6, label='Submission 2 (Uniform Rank Space)', color='#ff7f0e', density=True)
axes[0].set_title('Prediction Density Comparison', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Value')
axes[0].set_ylabel('Density')
axes[0].legend(frameon=True)

# Plot 2: Scatter plot sample
np.random.seed(42)
sample_idx = np.random.choice(len(df1), size=20000, replace=False)
axes[1].scatter(df1['Will_Buy_EV'].iloc[sample_idx], df2['Will_Buy_EV'].iloc[sample_idx], alpha=0.12, s=5, color='#2ca02c')
axes[1].plot([0, 1], [0, 1], 'r--', lw=1.5, label='Identity Line (y=x)')
axes[1].set_title('Submission 1 vs Submission 2 (20k Samples)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Submission 1 Value')
axes[1].set_ylabel('Submission 2 Value')
axes[1].legend(frameon=True)

plt.tight_layout()
plt.show()


### Rank Discrepancy Analysis
Measuring how much the relative ranks differ between the two models.


In [ ]:
n = len(df1)
r1 = rankdata(df1['Will_Buy_EV']) / n
r2 = rankdata(df2['Will_Buy_EV']) / n
rank_diff = np.abs(r1 - r2)

print(f"Mean Rank Difference:         {rank_diff.mean():.6f}")
print(f"Median Rank Difference:       {np.median(rank_diff):.6f}")
print(f"90th Percentile Difference:   {np.percentile(rank_diff, 90):.6f}")
print(f"99th Percentile Difference:   {np.percentile(rank_diff, 99):.6f}")
print(f"Maximum Rank Difference:      {rank_diff.max():.6f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(rank_diff, bins=100, color='#8c564b', alpha=0.8, log=True)
ax.set_title('Log-Scale Distribution of Absolute Normalized Rank Differences |r1 - r2|', fontsize=12, fontweight='bold')
ax.set_xlabel('|r1 - r2|')
ax.set_ylabel('Count (Log Scale)')
plt.show()


## 4. Multi-Strategy Blending Framework

We construct multiple ensembling candidates optimized for ROC AUC:

1. **Rank Average (50/50)**:
   $$S_{\text{rank\_50}} = 0.50 \cdot r_1 + 0.50 \cdot r_2$$
   *The gold standard for ROC AUC competitions: neutralizes calibration differences and reduces prediction variance.*

2. **Weighted Rank Average (55/45)**:
   $$S_{\text{rank\_55}} = 0.55 \cdot r_1 + 0.45 \cdot r_2$$
   *Slightly weights Submission 1 (giving higher weight to the calibrated probability model).*

3. **Logit-Space Rank Blend (50/50)**:
   $$z_1 = \text{logit}(\text{clip}(r_1, \epsilon, 1-\epsilon)), \quad z_2 = \text{logit}(\text{clip}(r_2, \epsilon, 1-\epsilon))$$
   $$z_{\text{blend}} = 0.50 \cdot z_1 + 0.50 \cdot z_2, \quad S_{\text{logit}} = \sigma(z_{\text{blend}})$$
   *Stretches the probability tails, providing high discrimination at both low and high thresholds.*

4. **Power-Rank Blend ($p=1.5$)**:
   $$S_{\text{power}} = 0.50 \cdot (r_1)^{1.5} + 0.50 \cdot (r_2)^{1.5}$$


In [ ]:
# 1. Rank Average (50/50)
blend_rank_50 = 0.50 * r1 + 0.50 * r2

# 2. Weighted Rank Average (55/45)
blend_rank_55 = 0.55 * r1 + 0.45 * r2

# 3. Weighted Rank Average (45/55)
blend_rank_45 = 0.45 * r1 + 0.55 * r2

# 4. Logit-Space Rank Blend (50/50)
eps = 1e-6
r1_c = np.clip(r1, eps, 1.0 - eps)
r2_c = np.clip(r2, eps, 1.0 - eps)
l1 = logit(r1_c)
l2 = logit(r2_c)
blend_logit_50 = expit(0.50 * l1 + 0.50 * l2)
blend_logit_55 = expit(0.55 * l1 + 0.45 * l2)

# 5. Power-Rank Blend (p=1.5)
blend_power_15 = 0.50 * (r1 ** 1.5) + 0.50 * (r2 ** 1.5)

print("[PASS] All 6 blend variations generated successfully!")


## 5. Blend Diagnostic Alignment Matrix
We verify that every blend exhibits near-perfect, balanced correlation with both source models.


In [ ]:
blend_dict = {
    'Sub 1 (Prob)': df1['Will_Buy_EV'].values,
    'Sub 2 (Rank)': df2['Will_Buy_EV'].values,
    'Rank Avg (50/50)': blend_rank_50,
    'Rank Avg (55/45)': blend_rank_55,
    'Rank Avg (45/55)': blend_rank_45,
    'Logit Blend (50/50)': blend_logit_50,
    'Logit Blend (55/45)': blend_logit_55,
    'Power Rank (p=1.5)': blend_power_15,
}

rows = []
for name, preds in blend_dict.items():
    s1_corr, _ = spearmanr(preds, df1['Will_Buy_EV'])
    s2_corr, _ = spearmanr(preds, df2['Will_Buy_EV'])
    rows.append({
        'Model / Blend Strategy': name,
        'Spearman vs Sub1': f"{s1_corr:.7f}",
        'Spearman vs Sub2': f"{s2_corr:.7f}",
        'Mean': f"{np.mean(preds):.6f}",
        'Std Dev': f"{np.std(preds):.6f}",
        'Min': f"{np.min(preds):.6e}",
        'Max': f"{np.max(preds):.6f}",
    })

eval_df = pd.DataFrame(rows)
print(eval_df.to_string(index=False))


## 6. Export Final Kaggle Submissions

We export the primary submission `submission.csv` (using the balanced **Rank Average (50/50)**), as well as supplementary blend files for Kaggle submission slot selection.


In [ ]:
# 1. Primary submission (Rank Average 50/50)
sub_primary = pd.DataFrame({
    'id': df1['id'],
    'Will_Buy_EV': blend_rank_50
})
sub_primary.to_csv('submission.csv', index=False)
print(f"[PRIMARY] Saved: submission.csv ({len(sub_primary):,} rows)")

# 2. Weighted Rank (55/45)
sub_w55 = pd.DataFrame({
    'id': df1['id'],
    'Will_Buy_EV': blend_rank_55
})
sub_w55.to_csv('submission_rank_55_45.csv', index=False)
print(f"[VARIANT] Saved: submission_rank_55_45.csv ({len(sub_w55):,} rows)")

# 3. Logit Blend (50/50)
sub_logit = pd.DataFrame({
    'id': df1['id'],
    'Will_Buy_EV': blend_logit_50
})
sub_logit.to_csv('submission_logit_50_50.csv', index=False)
print(f"[VARIANT] Saved: submission_logit_50_50.csv ({len(sub_logit):,} rows)")

# 4. Power Rank
sub_power = pd.DataFrame({
    'id': df1['id'],
    'Will_Buy_EV': blend_power_15
})
sub_power.to_csv('submission_power_rank.csv', index=False)
print(f"[VARIANT] Saved: submission_power_rank.csv ({len(sub_power):,} rows)")


## 7. Submission Integrity Verification
Ensuring all exported CSV files meet 100% of Kaggle formatting specifications.


In [ ]:
def verify_submission(filepath):
    df = pd.read_csv(filepath)
    assert df.shape == (286571, 2), f"Shape check failed! Found {df.shape}, expected (286571, 2)"
    assert list(df.columns) == ['id', 'Will_Buy_EV'], f"Column names check failed! Found {list(df.columns)}"
    assert df['id'].dtype in [np.int64, int], f"id is not integer: {df['id'].dtype}"
    assert df['Will_Buy_EV'].isna().sum() == 0, "Submission contains null/NaN values!"
    assert np.isinf(df['Will_Buy_EV']).sum() == 0, "Submission contains infinite values!"
    assert (df['Will_Buy_EV'] >= 0.0).all() and (df['Will_Buy_EV'] <= 1.0).all(), "Predictions outside [0, 1]!"
    assert (df['id'] == df1['id']).all(), "IDs do not match test order!"
    print(f"[PASS] Verified {filepath}: 286,571 rows | Min: {df['Will_Buy_EV'].min():.6f} | Max: {df['Will_Buy_EV'].max():.6f}")

verify_submission('submission.csv')
verify_submission('submission_rank_55_45.csv')
verify_submission('submission_logit_50_50.csv')
verify_submission('submission_power_rank.csv')

print("\n[SUCCESS] ALL VERIFICATION CHECKS PASSED SUCCESSFULLY! Ready to submit!")
